# Extracao de PST -> Excel para RAG Agentico (Agente de Suporte)

Pipeline:

1. **Extracao** do `dados-puc-campinas.pst` via MAPI estendido (aceita PST com senha), com Outlook COM e `pypff` como alternativas.
2. **Normalizacao / limpeza** do corpo (HTML -> texto, remocao de citacoes, assinaturas e disclaimers).
3. **Reconstrucao de threads** (agrupamento por conversa) e derivacao de pares **pergunta -> resposta**.
4. **Chunking** com overlap para indexacao vetorial.
5. **Export** para um unico `.xlsx` com as abas: `mensagens`, `threads`, `chunks_rag`, `estatisticas`.

O arquivo gerado e a fonte de ingestao do RAG: cada linha de `chunks_rag` vira um documento
com `chunk_id`, `texto` e metadados filtraveis (`thread_id`, `pasta`, `data`, `assunto`, `remetente`).

In [23]:
# 1. Dependencias
# pip install pandas openpyxl beautifulsoup4 lxml pywin32
# (Linux/macOS: pip install libpff-python)
#
# Sanitizacao (secao 4.5) - NER de nomes de pessoas:
#   pip install spacy && python -m spacy download pt_core_news_lg
#   (opcional: sem o modelo, a secao 4.5 segue so com a heuristica de saudacao/assinatura)

import os, re, sys, html, hashlib, unicodedata
from pathlib import Path
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup

print("python:", sys.version.split()[0], "| pandas:", pd.__version__)

python: 3.12.5 | pandas: 2.2.3


In [24]:
# 2. Configuracao
BASE_DIR   = Path.cwd()
PST_PATH   = (BASE_DIR / "dados_puc_digital_email.pst").resolve()
OUT_XLSX   = (BASE_DIR / "base_conhecimento_suporte.xlsx").resolve()

# Senha do PST (o export do Outlook foi protegido). Deixe "" se o arquivo nao tiver senha.
SENHA_PST = "silva007"

# Dominios internos = equipe de suporte. Mensagem vinda de fora e "pergunta do usuario";
# mensagem vinda de dentro e "resposta do suporte".
DOMINIOS_INTERNOS = ("puc-campinas.edu.br", "puccampinas.edu.br")

CHUNK_SIZE      = 1200   # caracteres por chunk (~300 tokens)
CHUNK_OVERLAP   = 150
MIN_CHARS_CHUNK = 80     # descarta ruido

# Modelos de resposta padrao (secao 5.5): respostas quase identicas que o suporte
# reenvia varias vezes viram UM documento canonico tipo "modelo".
MIN_OCORRENCIAS_MODELO = 3      # nº de respostas no cluster p/ virar modelo
SIMILARIDADE_MODELO    = 0.90   # merge de respostas com esqueleto parecido (0-1)

# Modelos cujo contexto casa CONTEXTOS_REDIRECIONAR tem o texto trocado por um
# encaminhamento a Secretaria Academica (datas de cronograma ficam desatualizadas).
CONTEXTOS_REDIRECIONAR = ("cronograma",)   # fragmentos regex; ex.: + "calend.rio acad"
EMAIL_SECRETARIA       = "puc.digital@puc-campinas.edu.br"

# =========================================================================================
#  Sanitizacao LGPD / GDPR  (ver secao 4.5)
#
#  A base RAG so serve de APOIO para a IA simular o agente de suporte: ela precisa dos
#  PROCEDIMENTOS, nunca da identidade de quem abriu o chamado. Postura = minimizacao de
#  dados (LGPD Art. 6, III; GDPR Art. 5(1)(c)) + protection by design & by default
#  (LGPD Art. 46; GDPR Art. 25).
# =========================================================================================
SANITIZAR          = True         # False => base em claro (proibido sair do ambiente controlado)
MODO_SANITIZACAO   = "rotulo"      # "rotulo"     -> [CPF]  (anonimizacao; recomendado p/ RAG)
                                   # "pseudonimo" -> [CPF:ab12cd] (hash HMAC estavel; so p/ correlacao)
USAR_NER           = True          # spaCy pt_core_news_lg p/ nomes de pessoas (cai p/ heuristica se ausente)
QUARENTENA_SENSIVEL = True         # dado sensivel (LGPD Art. 11 / GDPR Art. 9) -> fora do RAG
SANITIZAR_ESTRITO  = True          # aborta o export se sobrar PII na checagem anti-vazamento

assert PST_PATH.exists(), f"PST nao encontrado: {PST_PATH}"
print("PST :", PST_PATH, f"({PST_PATH.stat().st_size/1024:.1f} KB)")
print("XLSX:", OUT_XLSX)

PST : C:\Users\henrique.cordeiro\OneDrive - Sociedade Campineira de Educação e Instrução\python\Ler_dados_exportados_email\dados_puc_digital_email.pst (9304977.0 KB)
XLSX: C:\Users\henrique.cordeiro\OneDrive - Sociedade Campineira de Educação e Instrução\python\Ler_dados_exportados_email\base_conhecimento_suporte.xlsx


## 3. Extracao das mensagens

O caminho principal e o **MAPI estendido** (`extrair_mapi`): ele monta o PST num **perfil MAPI
temporario**, passando a senha em `PR_PST_PW_SZ_OLD`, le as mensagens e apaga o perfil no final.
E o unico caminho que funciona com PST protegido por senha — o `AddStore` do modelo de objetos do
Outlook recusa o arquivo em silencio ("nao foi carregado para esta sessao"), porque nao consegue
exibir o prompt de senha em automacao. O perfil padrao do Outlook nao e tocado.

In [25]:
# 3.1 Extrator principal: MAPI estendido (aceita PST com senha)
import pythoncom
from win32com.mapi import mapi, mapitags as T

PR_SENDER_SMTP    = 0x5D01001F   # PidTagSenderSmtpAddress
PR_SENT_REPR_SMTP = 0x5D02001F   # PidTagSentRepresentingSmtpAddress
PR_HTML           = 0x10130102   # PidTagHtml

TAGS_MSG = (T.PR_SUBJECT_W, T.PR_CONVERSATION_TOPIC_W, T.PR_SENDER_NAME_W,
            PR_SENDER_SMTP, PR_SENT_REPR_SMTP, T.PR_SENDER_EMAIL_ADDRESS_W,
            T.PR_DISPLAY_TO_W, T.PR_DISPLAY_CC_W, T.PR_CLIENT_SUBMIT_TIME,
            T.PR_MESSAGE_DELIVERY_TIME, T.PR_HASATTACH, T.PR_BODY_W)

def _s(v):
    if isinstance(v, bytes):
        return v.decode("utf-8", "ignore")
    return "" if v is None else str(v)

def _dt(v):
    try:
        return pd.to_datetime(str(v), errors="coerce")
    except Exception:
        return pd.NaT

def _linhas(tabela, tags):
    return [dict((p[0], p[1]) for p in r)
            for r in mapi.HrQueryAllRows(tabela, tags, None, None, 0)]

def _stream(obj, tag):
    """Le propriedade grande via IStream: corpos longos vem como PT_ERROR no GetProps."""
    try:
        st = obj.OpenProperty(tag, pythoncom.IID_IStream, 0, 0)
    except Exception:
        return ""
    partes = []
    while True:
        try:
            b = st.Read(65536)
        except Exception:
            break
        if not b:
            break
        partes.append(b)
    dados = b"".join(partes)
    if tag & 0xFFFF == 0x001F:            # PT_UNICODE
        return dados.decode("utf-16-le", "ignore")
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            return dados.decode(enc)
        except Exception:
            continue
    return dados.decode("utf-8", "ignore")

def _props(obj, tags):
    hr, pr = obj.GetProps(tags, 0)
    return {tag: (None if (tag & 0xFFFF) == T.PT_ERROR else val) for tag, val in pr}

def _ler_mensagem(store, entryid, caminho):
    msg = store.OpenEntry(entryid, None, 0)
    p = _props(msg, TAGS_MSG)
    corpo = p.get(T.PR_BODY_W) or _stream(msg, T.PR_BODY_W)
    anexos = []
    if p.get(T.PR_HASATTACH):
        try:
            for row in _linhas(msg.GetAttachmentTable(0), (T.PR_ATTACH_LONG_FILENAME_W,)):
                nome = row.get(T.PR_ATTACH_LONG_FILENAME_W)
                if isinstance(nome, str) and nome:
                    anexos.append(nome)
        except Exception:
            pass
    assunto = _s(p.get(T.PR_SUBJECT_W))
    return {
        "entry_id": entryid.hex()[:32],
        "pasta": caminho,
        "assunto": assunto,
        "conversa": _s(p.get(T.PR_CONVERSATION_TOPIC_W)) or assunto,
        "conversation_id": "",
        "remetente_nome": _s(p.get(T.PR_SENDER_NAME_W)),
        "remetente": _s(p.get(PR_SENDER_SMTP) or p.get(PR_SENT_REPR_SMTP)
                        or p.get(T.PR_SENDER_EMAIL_ADDRESS_W)).lower(),
        "para": _s(p.get(T.PR_DISPLAY_TO_W)),
        "cc": _s(p.get(T.PR_DISPLAY_CC_W)),
        "data_envio": _dt(p.get(T.PR_CLIENT_SUBMIT_TIME)),
        "data_recebido": _dt(p.get(T.PR_MESSAGE_DELIVERY_TIME)),
        "categorias": "",
        "anexos": "; ".join(anexos),
        "n_anexos": len(anexos),
        "corpo_html": _stream(msg, PR_HTML),
        "corpo_txt": corpo,
    }

def extrair_mapi(pst_path: Path, senha="", perfil="tmp_pst_rag", limite=None, verbose=True):
    mapi.MAPIInitialize(None)
    admin = mapi.MAPIAdminProfiles(0)
    try:
        admin.DeleteProfile(perfil, 0)      # resto de execucao anterior
    except Exception:
        pass
    admin.CreateProfile(perfil, None, 0, 0)
    registros = []
    try:
        session = mapi.MAPILogonEx(
            0, perfil, None, mapi.MAPI_EXTENDED | mapi.MAPI_NEW_SESSION | mapi.MAPI_NO_MAIL)
        svc = session.AdminServices(0)
        svc.CreateMsgService("MSUPST MS", "PST RAG", 0, 0)   # provedor de PST unicode
        linhas_svc = _linhas(svc.GetMsgServiceTable(0), (T.PR_SERVICE_UID,))
        uid = pythoncom.MakeIID(linhas_svc[-1][T.PR_SERVICE_UID], True)
        cfg = [(T.PR_PST_PATH_A, str(pst_path))]
        if senha:
            cfg += [(T.PR_PST_PW_SZ_OLD_A, senha), (T.PR_PST_REMEMBER_PW, True)]
        svc.ConfigureMsgService(uid, 0, 0, tuple(cfg))

        for linha in _linhas(session.GetMsgStoresTable(0), (T.PR_ENTRYID, T.PR_DISPLAY_NAME_W)):
            store = session.OpenMsgStore(0, linha[T.PR_ENTRYID], None,
                                         mapi.MDB_NO_DIALOG | mapi.MAPI_BEST_ACCESS)
            hr, pr = store.GetProps((T.PR_IPM_SUBTREE_ENTRYID,), 0)

            def walk(folder_eid, caminho):
                folder = store.OpenEntry(folder_eid, None, 0)
                for row in _linhas(folder.GetContentsTable(0), (T.PR_ENTRYID,)):
                    if limite and len(registros) >= limite:
                        return
                    try:
                        registros.append(_ler_mensagem(store, row[T.PR_ENTRYID], caminho))
                    except Exception as e:
                        print(f"  [skip] {caminho}: {e}")
                if verbose and caminho:
                    print(f"  {caminho}: {len(registros)} mensagens acumuladas")
                for row in _linhas(folder.GetHierarchyTable(0),
                                   (T.PR_ENTRYID, T.PR_DISPLAY_NAME_W)):
                    if limite and len(registros) >= limite:
                        return
                    nome = _s(row.get(T.PR_DISPLAY_NAME_W))
                    walk(row[T.PR_ENTRYID], f"{caminho}/{nome}" if caminho else nome)

            walk(pr[0][1], "")
    finally:
        try:
            admin.DeleteProfile(perfil, 0)   # nao deixa lixo no MAPI
        except Exception as e:
            print("aviso: falha ao remover o perfil temporario:", e)
        mapi.MAPIUninitialize()
    return registros

### 3.2 Alternativa: modelo de objetos do Outlook

Util quando o PST **nao** tem senha, ou quando voce ja abriu o arquivo manualmente no Outlook.
`extrair_outlook` tenta, nesta ordem:

1. **PST ja aberto no Outlook** — se o arquivo estiver montado no perfil, so percorre as pastas.
   Este e o caminho a usar quando o PST tem **senha**: o Outlook nao consegue pedir a senha via
   automacao, entao abra uma vez pela interface (Arquivo > Abrir e Exportar > Abrir Arquivo de
   Dados do Outlook) e rode a celula.
2. **Montagem automatica** (`AddStore`) — copiando antes para uma pasta local, porque o Outlook
   recusa PST dentro do OneDrive (o arquivo sincronizado e um *reparse point*). O store montado
   por aqui e desmontado ao final; um PST aberto por voce nao e mexido.

`extrair_pypff` e o fallback multiplataforma (em Windows exige compilador C, entao normalmente
o caminho valido e o Outlook).

In [26]:
# 3.3 Extrator via Outlook COM (Windows)
OL_MAIL = 43  # olMail

def _dt(v):
    try:
        return pd.to_datetime(str(v), errors="coerce")
    except Exception:
        return pd.NaT

def _prop(item, nome, default=""):
    try:
        v = getattr(item, nome)
        return default if v is None else v
    except Exception:
        return default

def _smtp(item):
    # Exchange devolve X500 em SenderEmailAddress; resolve para o SMTP real.
    addr = str(_prop(item, "SenderEmailAddress"))
    if addr.startswith("/") or addr.upper().startswith("EX:"):
        try:
            sender = item.Sender
            if sender is not None:
                ex = sender.GetExchangeUser()
                if ex is not None and ex.PrimarySmtpAddress:
                    return ex.PrimarySmtpAddress
        except Exception:
            pass
        try:
            return item.PropertyAccessor.GetProperty(
                "http://schemas.microsoft.com/mapi/proptag/0x39FE001E")
        except Exception:
            return addr
    return addr

def _walk(folder, caminho, registros):
    caminho = f"{caminho}/{folder.Name}" if caminho else folder.Name
    try:
        itens = folder.Items
    except Exception:
        itens = []
    for i in range(1, getattr(itens, "Count", 0) + 1):
        try:
            it = itens.Item(i)
            if _prop(it, "Class", 0) != OL_MAIL:
                continue
            anexos = []
            try:
                for k in range(1, it.Attachments.Count + 1):
                    anexos.append(it.Attachments.Item(k).FileName)
            except Exception:
                pass
            registros.append({
                "entry_id":        str(_prop(it, "EntryID")),
                "pasta":           caminho,
                "assunto":         str(_prop(it, "Subject")),
                "conversa":        str(_prop(it, "ConversationTopic")) or str(_prop(it, "Subject")),
                "conversation_id": str(_prop(it, "ConversationID")),
                "remetente_nome":  str(_prop(it, "SenderName")),
                "remetente":       str(_smtp(it)).lower(),
                "para":            str(_prop(it, "To")),
                "cc":              str(_prop(it, "CC")),
                "data_envio":      _dt(_prop(it, "SentOn", None)),
                "data_recebido":   _dt(_prop(it, "ReceivedTime", None)),
                "categorias":      str(_prop(it, "Categories")),
                "anexos":          "; ".join(anexos),
                "n_anexos":        len(anexos),
                "corpo_html":      str(_prop(it, "HTMLBody")),
                "corpo_txt":       str(_prop(it, "Body")),
            })
        except Exception as e:
            print(f"  [skip] {caminho} item {i}: {e}")
    for j in range(1, folder.Folders.Count + 1):
        _walk(folder.Folders.Item(j), caminho, registros)

def _namespace():
    import win32com.client
    ns = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
    try:
        ns.Logon("", "", False, False)
    except Exception:
        pass
    return ns

def listar_stores(ns=None):
    """Mostra os arquivos de dados abertos no Outlook (util para diagnostico)."""
    ns = ns or _namespace()
    linhas = []
    for s in ns.Stores:
        try:
            linhas.append({"nome": s.DisplayName, "arquivo": s.FilePath or ""})
        except Exception as e:
            linhas.append({"nome": f"<erro: {e}>", "arquivo": ""})
    return pd.DataFrame(linhas)

def _achar_store(ns, pst_path: Path):
    """Procura o PST ja aberto no Outlook (por caminho completo ou nome do arquivo)."""
    alvo, nome = str(pst_path).lower(), pst_path.name.lower()
    for s in ns.Stores:
        try:
            fp = (s.FilePath or "").lower()
        except Exception:
            continue
        if fp and (fp == alvo or os.path.basename(fp) == nome):
            return s
    return None

def extrair_outlook(pst_path: Path):
    import shutil, tempfile

    ns = _namespace()
    tmp_dir, montado_por_nos = None, False

    # 1) PST ja aberto no Outlook (File > Abrir e Exportar > Abrir Arquivo de Dados do Outlook).
    store = _achar_store(ns, pst_path)
    if store is not None:
        print("Usando PST ja aberto no Outlook:", store.DisplayName)
    else:
        # 2) monta o PST na sessao. O Outlook recusa arquivo em pasta sincronizada
        #    (OneDrive marca o arquivo como reparse point), entao usa uma copia local.
        alvo = str(pst_path)
        if "onedrive" in alvo.lower() or alvo.startswith("\\\\"):
            tmp_dir = tempfile.mkdtemp(prefix="pst_")
            alvo = str(Path(tmp_dir) / pst_path.name)
            print("Copiando PST para pasta local (pode demorar):", alvo)
            shutil.copy2(pst_path, alvo)
        try:
            ns.AddStore(alvo)
            montado_por_nos = True
        except Exception as e:
            if tmp_dir:
                shutil.rmtree(tmp_dir, ignore_errors=True)
            raise RuntimeError(
                "O Outlook recusou o arquivo ('nao foi carregado para esta sessao').\n"
                "Causas usuais, em ordem de probabilidade:\n"
                "  1. PST protegido por senha -> o Outlook nao consegue pedir a senha via automacao;\n"
                "  2. PST precisando de reparo -> rode o ScanPST.exe "
                "(C:\\Program Files\\Microsoft Office\\root\\Office16\\SCANPST.EXE);\n"
                "  3. PST vazio ou gerado por outra ferramenta.\n"
                "SOLUCAO MAIS RAPIDA: abra o PST manualmente no Outlook "
                "(Arquivo > Abrir e Exportar > Abrir Arquivo de Dados do Outlook), digite a senha "
                "se for pedida, e reexecute esta celula - o notebook detecta o store ja aberto."
            ) from e
        store = _achar_store(ns, Path(alvo))
        if store is None:
            raise RuntimeError("PST montado mas store nao localizado no perfil MAPI.")

    root = store.GetRootFolder()
    registros = []
    try:
        for j in range(1, root.Folders.Count + 1):
            _walk(root.Folders.Item(j), "", registros)
    finally:
        if montado_por_nos:
            try:
                ns.RemoveStore(root)   # desmonta apenas o que este notebook montou
            except Exception as e:
                print("aviso: falha ao desmontar o PST:", e)
        if tmp_dir:
            shutil.rmtree(tmp_dir, ignore_errors=True)
    return registros

In [27]:
# 3.4 Fallback multiplataforma via pypff (libpff-python)
def extrair_pypff(pst_path: Path):
    import pypff
    pst = pypff.file()
    pst.open(str(pst_path))
    registros = []

    def walk(folder, caminho):
        nome = folder.get_name() or ""
        caminho = f"{caminho}/{nome}" if caminho else nome
        for i in range(folder.get_number_of_sub_messages()):
            m = folder.get_sub_message(i)
            def g(fn, default=""):
                try:
                    v = fn()
                    return default if v is None else v
                except Exception:
                    return default
            corpo_html = g(m.get_html_body, b"")
            if isinstance(corpo_html, bytes):
                corpo_html = corpo_html.decode("utf-8", "ignore")
            corpo_txt = g(m.get_plain_text_body, b"")
            if isinstance(corpo_txt, bytes):
                corpo_txt = corpo_txt.decode("utf-8", "ignore")
            assunto = str(g(m.get_subject))
            registros.append({
                "entry_id": f"{caminho}#{i}",
                "pasta": caminho,
                "assunto": assunto,
                "conversa": str(g(m.get_conversation_topic)) or assunto,
                "conversation_id": "",
                "remetente_nome": str(g(m.get_sender_name)),
                "remetente": "",
                "para": "", "cc": "",
                "data_envio": _dt(g(m.get_client_submit_time, None)),
                "data_recebido": _dt(g(m.get_delivery_time, None)),
                "categorias": "",
                "anexos": "",
                "n_anexos": g(m.get_number_of_attachments, 0),
                "corpo_html": corpo_html,
                "corpo_txt": corpo_txt,
            })
        for j in range(folder.get_number_of_sub_folders()):
            walk(folder.get_sub_folder(j), caminho)

    walk(pst.get_root_folder(), "")
    pst.close()
    return registros

In [28]:
# 3.5 Diagnostico: arquivos de dados abertos no Outlook
try:
    display(listar_stores())
except Exception as e:
    print("Outlook COM indisponivel:", e)

,nome,arquivo
0,henrique.cordeiro@puc-campinas.edu.br,C:\Users\henrique.cordeiro\AppData\Local\Micro...
1,suporte.canvas@puc-campinas.edu.br,C:\Users\henrique.cordeiro\AppData\Local\Micro...
2,puc.digital@puc-campinas.edu.br,C:\Users\henrique.cordeiro\AppData\Local\Micro...


In [29]:
# 3.6 Execucao da extracao (MAPI -> Outlook COM -> pypff)
import time

tentativas = [
    ("mapi",        lambda: extrair_mapi(PST_PATH, SENHA_PST)),
    ("outlook-com", lambda: extrair_outlook(PST_PATH)),
    ("pypff",       lambda: extrair_pypff(PST_PATH)),
]

registros, motor, erros = [], None, []
t0 = time.time()
for nome, fn in tentativas:
    try:
        print(f"--- tentando extrator: {nome}")
        registros = fn()
        motor = nome
        break
    except Exception as e:
        erros.append(f"[{nome}] {type(e).__name__}: {e}")
        print(f"    falhou: {type(e).__name__}: {e}\n")

if motor is None:
    raise RuntimeError("Nenhum extrator funcionou:\n" + "\n".join(erros))
print(f"\nmotor={motor} | {len(registros)} mensagens em {time.time()-t0:.0f}s")

df_raw = pd.DataFrame(registros)
if df_raw.empty:
    print(
        "\nATENCAO: nenhuma mensagem encontrada.\n"
        f"  - tamanho do PST: {PST_PATH.stat().st_size} bytes "
        "(271360 bytes = PST unicode vazio, a exportacao nao gravou itens)\n"
        "  - refaca o export no Outlook (Arquivo > Abrir e Exportar > Importar/Exportar) "
        "marcando 'Incluir subpastas' e aguarde a conclusao."
    )
else:
    display(df_raw["pasta"].value_counts().to_frame("mensagens"))
    display(df_raw.head(3)[["pasta", "assunto", "remetente", "data_envio"]])

--- tentando extrator: mapi
  Itens Excluídos: 0 mensagens acumuladas
  Itens Enviados: 17818 mensagens acumuladas
  Caixa de Entrada: 17818 mensagens acumuladas
  Caixa de Saída: 17818 mensagens acumuladas


Win32 exception occurred releasing IUnknown at 0x000001B90A610F40



motor=mapi | 17818 mensagens em 213s


,mensagens
pasta,
Itens Enviados,17818


,pasta,assunto,remetente,data_envio
0,Itens Enviados,RES: Acesso ao Canvas,/o=exchangelabs/ou=exchange administrative gro...,2024-03-11 23:40:52.963000+00:00
1,Itens Enviados,Aviso CANVAS,/o=exchangelabs/ou=exchange administrative gro...,2023-05-31 17:06:45.469000+00:00
2,Itens Enviados,RES: Especialização em Machine Learning,/o=exchangelabs/ou=exchange administrative gro...,2021-08-10 11:23:32.029000+00:00


## 4. Limpeza, correcao de encoding e separacao resposta / citacao

Tres coisas precisam acontecer aqui, e a ordem importa:

1. **Encoding**: cerca de metade das mensagens vem com UTF-8 interpretado como cp1252
   (`prazo Ã© de 120 dias`). O `corrigir_mojibake` faz o *round-trip* `cp1252 -> utf-8` e so
   adota o resultado se ele reduzir as sequencias suspeitas.
2. **Separacao**: este PST contem apenas **Itens Enviados** — cada mensagem e uma resposta do
   suporte com a conversa original citada abaixo. Entao o corpo e cortado no primeiro cabecalho
   de citacao (`De:/From:` + `Enviada em:/Para:`, linha de `____`, ou `... escreveu:`):
   o texto acima e a **resposta do suporte**, o de baixo e o **historico**.
3. **Pergunta**: o primeiro bloco do historico e a mensagem que motivou a resposta, ou seja,
   a **pergunta do usuario**. E dela que sai o par pergunta -> resposta do RAG.

In [30]:
# 4. Limpeza
RE_CITACAO = re.compile(
    r"(?im)^[ \t>_-]*(?:de|from)\s*:\s*.{0,200}$\s*"
    r"(?:^[ \t>]*(?:enviad[oa](?:\s*em)?|sent|data|date)\s*:\s*.{0,120}$\s*)?"
    r"(?:^[ \t>]*(?:para|to)\s*:\s*.{0,300}$\s*)?"
)
RE_ESCREVEU   = re.compile(r"(?im)^.{0,160}?\b(escreveu|wrote)\s*:\s*$")
RE_SEP_LINHA  = re.compile(r"(?m)^[ \t]*_{10,}[ \t]*$")
RE_CAB_SOLTO  = re.compile(
    r"(?im)^[ \t>]*(?:de|from|para|to|cc|cco|bcc|enviad[oa](?:\s*em)?|sent|data|date|assunto|subject)\s*:.*$")
RE_DISCLAIMER = re.compile(
    r"(?is)(?:aviso legal|esta mensagem.{0,40}confidencial|this (e-?mail|message).{0,60}confidential|"
    r"antes de imprimir|pense no meio ambiente).*$")
RE_ASSINATURA = re.compile(r"(?m)^\s*--\s*$")
RE_URL   = re.compile(r"https?://\S+")
RE_MAILTO = re.compile(r"\s*<mailto:[^>]*>")
RE_WS    = re.compile(r"[ \t\xa0]+")
RE_NL    = re.compile(r"\n{3,}")

# mensagens automaticas que nao ensinam nada ao agente de suporte
RE_RUIDO = re.compile(
    r"(?i)(?:ingressar na reuni[ao]o agora|join the meeting now|id da reuni[ao]o|"
    r"microsoft teams.{0,30}precisa de ajuda|op[cç][oõ]es de reuni[ao]o|"
    r"esta mensagem foi enviada automaticamente|no-?reply)")

MARCAS_MOJIBAKE = ("Ã", "Â", "â€")

def _corrigir_token(tk: str) -> str:
    """Round-trip estrito por palavra.

    Estrito de proposito: com errors="ignore" a correcao seria destrutiva, apagando os acentos
    que ja estavam certos. Por palavra (e nao por linha) porque um emoji ou outro caractere fora
    do cp1252 em qualquer ponto da linha faria a linha inteira ficar sem conserto.
    """
    if not any(m in tk for m in MARCAS_MOJIBAKE):
        return tk
    try:
        return tk.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return tk

def corrigir_mojibake(t: str) -> str:
    """UTF-8 lido como cp1252 ('Ã©' em vez de 'e' acentuado) -> texto correto."""
    if not t or not any(m in t for m in MARCAS_MOJIBAKE):
        return t
    t = t.replace("\r\n", "\n")
    return "".join(_corrigir_token(tk) for tk in re.split(r"(\s+)", t))

def html_para_texto(h: str) -> str:
    if not h or not h.strip():
        return ""
    try:
        soup = BeautifulSoup(h, "lxml")
    except Exception:
        soup = BeautifulSoup(h, "html.parser")
    for tag in soup(["script", "style", "head", "meta", "title"]):
        tag.decompose()
    for tag in soup.find_all(["br", "p", "div", "tr", "li"]):
        tag.append("\n")
    return html.unescape(soup.get_text(" "))

def normaliza(t: str) -> str:
    t = corrigir_mojibake(t or "")
    t = unicodedata.normalize("NFKC", t).replace("\r\n", "\n").replace("\r", "\n")
    t = t.replace("\xad", "")            # soft hyphen que o Outlook injeta
    t = RE_MAILTO.sub("", t)
    t = RE_WS.sub(" ", t)
    t = "\n".join(l.strip() for l in t.split("\n"))
    return RE_NL.sub("\n\n", t).strip()

def corpo_bruto(row) -> str:
    base = row.get("corpo_txt") or ""
    if len(base.strip()) < 30 and row.get("corpo_html"):
        base = html_para_texto(row["corpo_html"])
    return base

def separar(texto: str):
    """(resposta escrita pelo suporte, historico citado abaixo dela)."""
    t = normaliza(texto)
    cortes = [m.start() for m in (RE_CITACAO.search(t), RE_ESCREVEU.search(t),
                                  RE_SEP_LINHA.search(t)) if m]
    if not cortes:
        return t, ""
    i = min(cortes)
    return t[:i].strip(), t[i:].strip()

def limpar_bloco(b: str) -> str:
    b = RE_CITACAO.sub("", b)
    b = RE_CAB_SOLTO.sub("", b)       # sobras de 'Enviado:/Para:/Assunto:'
    b = RE_SEP_LINHA.sub("", b)
    b = RE_ESCREVEU.sub("", b)
    b = RE_DISCLAIMER.sub("", b)
    partes = RE_ASSINATURA.split(b)
    if len(partes) > 1 and len(partes[0].strip()) > 40:
        b = partes[0]
    return normaliza(b)

def blocos_historico(hist: str):
    """Historico citado quebrado em mensagens, da mais recente para a mais antiga."""
    marcas = sorted(set([m.start() for m in RE_CITACAO.finditer(hist)] +
                        [m.end() for m in RE_ESCREVEU.finditer(hist)]))
    if not marcas:
        b = limpar_bloco(hist)
        return [b] if b else []
    blocos = []
    for ini, fim in zip(marcas, marcas[1:] + [len(hist)]):
        b = limpar_bloco(hist[ini:fim])
        if len(b) >= 15:
            blocos.append(b)
    return blocos

def norm_assunto(s: str) -> str:
    s = re.sub(r"(?i)^\s*((re|res|enc|fw|fwd|encaminhada)\s*:\s*)+", "", str(s or "")).strip()
    return RE_WS.sub(" ", s).lower()

def eh_interno(row) -> bool:
    e = (row.get("remetente") or "").lower()
    if any(d in e for d in DOMINIOS_INTERNOS):
        return True
    if e.startswith("/o="):                       # endereco X500 do Exchange = conta interna
        return True
    return "enviado" in (row.get("pasta") or "").lower()

if df_raw.empty:
    raise SystemExit("PST sem mensagens legiveis - verifique o arquivo.")

df = df_raw.copy()
_partes = df.apply(lambda r: separar(corpo_bruto(r)), axis=1)
df["resposta"]  = [p[0] for p in _partes]
df["historico"] = [p[1] for p in _partes]
df["blocos"]    = df["historico"].map(blocos_historico)
df["pergunta"]  = df["blocos"].map(lambda b: b[0] if b else "")
df["resposta"]  = df["resposta"].map(limpar_bloco)

# origem e' derivada ANTES da sanitizacao (precisa do e-mail/dominio real do remetente).
df["data"]        = df["data_envio"].fillna(df["data_recebido"])
df["origem"]      = df.apply(lambda r: "suporte" if eh_interno(r) else "usuario", axis=1)
df["assunto"]     = df["assunto"].map(lambda s: normaliza(s))
df["assunto_norm"]= df["conversa"].where(df["conversa"].str.strip().ne(""), df["assunto"]).map(norm_assunto)
df["thread_id"]   = df.apply(
    lambda r: (r["conversation_id"] or "").strip()
    or "T" + hashlib.md5((r["assunto_norm"] or r["entry_id"]).encode("utf-8")).hexdigest()[:12],
    axis=1)
df["n_chars"]     = df["resposta"].str.len()
df["n_palavras"]  = df["resposta"].str.split().str.len().fillna(0).astype(int)
df["tem_pergunta"]= df["pergunta"].str.len().ge(20)

antes = len(df)
df = df[df["n_chars"] >= 40]                                        # respostas vazias/so assinatura
df = df[~df["resposta"].str.contains(RE_RUIDO, na=False)]           # convites de reuniao etc.
df["_dedup"] = (df["assunto_norm"] + "|" + df["resposta"]).map(
    lambda s: hashlib.md5(s.encode("utf-8")).hexdigest())
df = df.drop_duplicates("_dedup").drop(columns=["_dedup"])
df = df.sort_values(["thread_id", "data"], kind="stable").reset_index(drop=True)
df["msg_id"] = ["M{:05d}".format(i + 1) for i in range(len(df))]

# >>> A sanitizacao de dados sigilosos (CPF/RG/RA/e-mail/telefone) roda na secao 4.5,
#     logo abaixo, antes de montar threads (secao 5) e chunks (secao 6).

print(f"{antes} -> {len(df)} mensagens uteis | {df['thread_id'].nunique()} threads | "
      f"{int(df['tem_pergunta'].sum())} com pergunta identificada")
display(df[["msg_id", "assunto", "origem", "n_palavras", "tem_pergunta"]].head(8))
print("\n--- exemplo de par ---")
_ex = df[df["tem_pergunta"]].iloc[0]
print("PERGUNTA:", _ex["pergunta"][:300], "\n\nRESPOSTA:", _ex["resposta"][:300])

17818 -> 13291 mensagens uteis | 5342 threads | 10965 com pergunta identificada


,msg_id,assunto,origem,n_palavras,tem_pergunta
0,M00001,ENC: Pós-graduação em Geriatria e Gerontologia...,suporte,67,True
1,M00002,RES: Adição Errada em Salas no CANVAS,suporte,21,True
2,M00003,RES: Ajuda com acesso_Matrícula 23029784 Marle...,suporte,463,True
3,M00004,RE: Problemas com notas,suporte,18,True
4,M00005,RES: Problemas com notas,suporte,17,True
5,M00006,RES: Problemas com notas,suporte,22,True
6,M00007,RES: Data de Vencimentos dos Boletos,suporte,31,True
7,M00008,TUTORIAL CANVAS ACESSO POR CELULAR-CURSO DE ES...,suporte,43,False



--- exemplo de par ---
PERGUNTA: Boa tarde, tudo bem?

Meu nome é Luiz Felipe, CPF 413.606.578-78, e gostaria de saber como devi proceder com a minha matrícula do curso EAD que foi cancelada, contudo eu já tinha efetuado o pagamento.

Eu gostaria de saber se tem a possibilidade de realizar uma matrícula extemporanea?

Aguardo conta 

RESPOSTA: Prezada Mariana, Roberto e Milena boa tarde

Espero encontra-los bem!

Venho por meio deste respeitosamente solicitar auxilio.

O contrato do candidato Luiz Felipe Favaro Batista RA: 23022234 do curso de especialização em Geriatria e Gerontologia foi cancelado, seria possível encaminhar um novo link


## 4.5 Sanitizacao LGPD / GDPR

**Finalidade da base (define o nivel de tratamento):** este `.xlsx` alimenta um RAG que serve
*apenas de apoio* para a IA simular o agente de suporte. O modelo precisa saber **como se
resolve** cada tipo de chamado — nunca **quem** pediu. Logo, todo dado pessoal e ruido: a
postura e de **minimizacao / anonimizacao** (LGPD Art. 6, III e Art. 12; GDPR Art. 5(1)(c),
Recital 26), aplicada *antes* de gerar threads e chunks (**protection by design & by default** —
LGPD Art. 46; GDPR Art. 25).

### Camadas

| # | Camada | Alvo |
|---|---|---|
| 1 | **Regex + digito verificador** | CPF, CNPJ, cartao (Luhn), RG, RA/matricula, CNH, PIS/NIS, titulo de eleitor, CNS, e-mail, telefone, CEP, IP, placa, IBAN/conta, data de nascimento, senha/token, sequencia numerica longa |
| 2 | **NER (spaCy) + heuristica de saudacao/assinatura** | nomes de pessoas |
| 3 | **Papel no lugar da identidade** | remetente/destinatario viram `[ATENDENTE]` / `[USUARIO]`; e-mail do remetente e descartado (ja cumpriu o papel de definir `origem`) |
| 4 | **Varredura de categoria especial** (LGPD Art. 11 / GDPR Art. 9) | saude, laudo/CID, deficiencia, religiao, raca/etnia, filiacao sindical/partidaria, orientacao sexual, biometria → linha vai para **quarentena** e sai do RAG |
| 5 | **Checagem anti-vazamento** | procura CPF/e-mail/telefone residual; com `SANITIZAR_ESTRITO=True` **aborta o export** |

### Modos (`MODO_SANITIZACAO`)

- **`rotulo`** — `[CPF]`, `[RG]`, `[EMAIL]`, `[NOME]`… Irreversivel. **Recomendado**: o vetor
  indexado nao carrega nada re-identificavel.
- **`pseudonimo`** — `[CPF:ab12cd]` (HMAC-SHA256 + salt de sessao nao persistido). Mantem
  correlacao "mesmo titular" dentro da execucao. Continua sendo **dado pessoal pseudonimizado**
  (GDPR Recital 26) — use so se a correlacao for realmente necessaria e a base ficar no ambiente
  interno. Nomes e enderecos ignoram esse modo e vao sempre a `[NOME]` / `[ENDERECO]`.

### Saidas de auditoria

- Aba `sanitizacao_pii` — contagem de ocorrencias mascaradas por tipo.
- Aba `quarentena_log` — `msg_id` + termo sensivel detectado (sem o corpo).
- `quarentena_revisao_manual.csv` (arquivo separado, **nao** e o RAG) — mensagens retiradas,
  para tratamento manual pelo/a encarregado/a (DPO).
- Aba `estatisticas` — flags `sanitizado`, `modo_sanitizacao`, `mensagens_quarentena`.

> **Risco residual (registrar no RIPD/DPIA):** texto livre pode re-identificar por combinacao de
> quase-identificadores (curso + campus + ano + evento raro). A automacao reduz, nao zera. Trate
> o vector store como contendo dado **pseudonimizado**, com controle de acesso e retencao
> definida, ate validacao manual amostral. O prompt do agente deve proibir solicitar ou repetir
> CPF/RG/RA e encaminhar ao canal oficial quando o dado for necessario.


In [ ]:
# 4.5 Sanitizacao LGPD / GDPR
import hmac, secrets

SAL_SANITIZACAO = os.environ.get("SAL_SANITIZACAO") or secrets.token_hex(16)
_CONTADOR_PII: dict = {}
_LOG_SENSIVEL: list = []
df_quarentena = df.iloc[0:0].copy()

# ----------------------------------------------------------------------------- validadores
def _digs(s):            # so digitos
    return re.sub(r"\D", "", s or "")

def _cpf_valido(v):
    n = _digs(v)
    if len(n) != 11 or n == n[0] * 11:
        return False
    for k in (9, 10):
        d = (sum(int(n[i]) * (k + 1 - i) for i in range(k)) * 10) % 11 % 10
        if d != int(n[k]):
            return False
    return True

def _cnpj_valido(v):
    n = _digs(v)
    if len(n) != 14 or n == n[0] * 14:
        return False
    pesos = [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]   # 13 -> fatia p/ 1o (12) e 2o (13) DV
    for tam in (12, 13):
        p = pesos[-tam:]
        d = 11 - (sum(int(n[i]) * p[i] for i in range(tam)) % 11)
        d = 0 if d >= 10 else d
        if d != int(n[tam]):
            return False
    return True

def _luhn_valido(v):
    n = _digs(v)
    if not (13 <= len(n) <= 19):
        return False
    soma, alt = 0, False
    for c in reversed(n):
        x = int(c) * (2 if alt else 1)
        soma += x - 9 if x > 9 else x
        alt = not alt
    return soma % 10 == 0

# ----------------------------------------------------------------------------- tokenizacao
_SEMPRE_ROTULO = {"NOME", "ENDERECO"}          # identidade direta: nunca pseudonimo

def _token(tag, valor=""):
    _CONTADOR_PII[tag] = _CONTADOR_PII.get(tag, 0) + 1
    if MODO_SANITIZACAO == "pseudonimo" and valor and tag not in _SEMPRE_ROTULO:
        chave = re.sub(r"[\s.\-/()]", "", str(valor)).lower().encode("utf-8")
        h = hmac.new(SAL_SANITIZACAO.encode("utf-8"), chave, "sha256").hexdigest()[:8]
        return f"[{tag}:{h}]"
    return f"[{tag}]"

# ----------------------------------------------------------- regras estruturadas (camada 1)
# tipo "ctx"     : grupo(1) = rotulo a preservar, grupo(2) = valor a mascarar
# tipo "simples" : match inteiro e o valor; guard(m) opcional decide se mascara
_CTX = [
    ("RA",  re.compile(r"(?i)(\b(?:R\.?\s?A\.?|registro\s+acad[eê]mico|matr[ií]cula|n[º°o]?\s*matr[ií]cula)\b[^\w\n]{0,8})(\d[\d.\-]{2,13}\d)"), None),
    ("RG",  re.compile(r"(?i)(\bR\.?\s?G\.?\b[^\w\n]{0,8})(\d[\d.\-xX]{4,12}[\dxX])"), None),
    ("CNH", re.compile(r"(?i)(\b(?:CNH|carteira\s+de\s+habilita[cç][aã]o)\b[^\w\n]{0,8})(\d{9,11})"), None),
    ("PIS", re.compile(r"(?i)(\b(?:PIS|PASEP|NIS|NIT)\b[^\w\n]{0,6})(\d{3}\.?\d{5}\.?\d{2}-?\d)"), None),
    ("TITULO_ELEITOR", re.compile(r"(?i)(\bt[ií]tulo(?:\s+de\s+eleitor)?\b[^\w\n]{0,6})(\d{4}\s?\d{4}\s?\d{4})"), None),
    ("CNS", re.compile(r"(?i)(\b(?:CNS|cart[aã]o\s+(?:nacional\s+de\s+sa[uú]de|sus))\b[^\w\n]{0,6})(\d[\d\s]{12,17}\d)"), None),
    ("DATA_NASC", re.compile(r"(?i)(\b(?:nascimento|nasc\.?|data\s+de\s+nasc(?:imento)?|d\.?n\.?)\b[^\w\n]{0,10})(\d{2}[/.\-]\d{2}[/.\-]\d{2,4})"), None),
    ("CONTA_BANCARIA", re.compile(r"(?i)(\b(?:ag[eê]ncia|ag\.?|conta(?:\s+corrente)?|c/c)\b[^\w\n]{0,8})(\d[\d.\-]{3,12}\d)"), None),
    ("CREDENCIAL", re.compile(r"(?i)(\b(?:senha|password|token|api[_\s\-]?key|secret|bearer)\b\s*[:=]\s*)(\S{4,})"), None),
    ("PROTOCOLO", re.compile(r"(?i)(\b(?:protocolo|chamado|ticket|n[º°o]?\s*(?:do\s+)?atendimento)\b[^\w\n]{0,6})([A-Za-z0-9][A-Za-z0-9\-/.]{4,20})"), None),
]
_SIMPLES = [
    ("EMAIL",    re.compile(r"(?i)\b[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}\b"), None),
    ("CPF",      re.compile(r"(?<!\d)\d{3}\.\d{3}\.\d{3}-\d{2}(?!\d)"), None),
    ("CPF",      re.compile(r"(?<!\d)\d{11}(?!\d)"), lambda m: _cpf_valido(m.group(0))),
    ("CNPJ",     re.compile(r"(?<!\d)\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}(?!\d)"), lambda m: _cnpj_valido(m.group(0))),
    ("CARTAO",   re.compile(r"(?<!\d)\d{4}[ \-]?\d{4}[ \-]?\d{4}[ \-]?\d{1,7}(?!\d)"), lambda m: _luhn_valido(m.group(0))),
    ("IBAN",     re.compile(r"\bBR\d{2}[A-Z0-9]{20,25}\b"), None),
    ("RG",       re.compile(r"(?<!\d)\d{1,2}\.\d{3}\.\d{3}-?[\dxX](?!\w)"), None),
    ("TELEFONE", re.compile(r"(?<!\d)(?:\+?55[\s.\-]?)?\(?\d{2}\)?[\s.\-]?9?\d{4}[\s.\-]?\d{4}(?!\d)"), None),
    ("CEP",      re.compile(r"(?<!\d)\d{5}-\d{3}(?!\d)"), None),
    ("IP",       re.compile(r"(?<!\d)(?:\d{1,3}\.){3}\d{1,3}(?!\d)"), lambda m: all(o.isdigit() and int(o) <= 255 for o in m.group(0).split("."))),
    ("PLACA",    re.compile(r"(?<![A-Za-z0-9])[A-Z]{3}[-\s]?\d[A-Z0-9]\d{2}(?![A-Za-z0-9])"), None),
    ("SEQ_NUMERICA", re.compile(r"(?<!\d)\d{13,}(?!\d)"), None),
]

def mascara_estruturada(t):
    if not isinstance(t, str) or not t:
        return t
    for tag, rx, guard in _CTX:
        t = rx.sub(lambda m, tag=tag, g=guard: (m.group(0) if (g and not g(m))
                                                else m.group(1) + _token(tag, m.group(2))), t)
    for tag, rx, guard in _SIMPLES:
        t = rx.sub(lambda m, tag=tag, g=guard: (m.group(0) if (g and not g(m))
                                                else _token(tag, m.group(0))), t)
    return t

# --------------------------------------------------------------- nomes de pessoas (camada 2)
_nlp = None
if SANITIZAR and USAR_NER:
    try:
        import spacy
        for _m in ("pt_core_news_lg", "pt_core_news_md", "pt_core_news_sm"):
            try:
                _nlp = spacy.load(_m, disable=["lemmatizer", "tagger", "parser", "attribute_ruler"])
                print("NER:", _m)
                break
            except Exception:
                continue
        if _nlp is None:
            print("NER: modelo pt nao instalado -> `python -m spacy download pt_core_news_lg` "
                  "(seguindo so com heuristica de saudacao/assinatura)")
    except Exception as e:
        print("NER: spacy indisponivel:", e)

_NOMES_STOP = {"senhor", "senhora", "aluno", "aluna", "professor", "professora", "prezado",
               "prezada", "coordenador", "coordenacao", "suporte", "equipe", "secretaria",
               "diretoria", "reitoria", "bom", "boa", "favor", "att", "atenciosamente", "obrigado"}
_RE_SAUDACAO = re.compile(
    r"(?im)\b(prezad[oa]s?|car[oa]|ol[aá]|senhor(?:a)?|sr\.?a?\.?)[\s,]+"
    r"([A-ZÀ-Ú][a-zà-ú]+(?:\s+[A-ZÀ-Ú][a-zà-ú]+){0,3})")
_RE_ASSIN = re.compile(
    r"(?im)^(at(?:enciosamente|\.?t?\.?)|abra[cç]os|grat[oa]|obrigad[oa]|cordialmente)[\s,]*\n+\s*"
    r"([A-ZÀ-Ú][a-zà-ú]+(?:\s+[A-ZÀ-Ú][a-zà-ú]+){0,3})")

def _rep_nome(m):
    if m.group(2).split()[0].lower() in _NOMES_STOP:
        return m.group(0)
    return m.group(0).replace(m.group(2), _token("NOME", m.group(2)), 1)

def mascara_nome_heuristica(t):
    if not isinstance(t, str) or not t:
        return t
    t = _RE_SAUDACAO.sub(_rep_nome, t)
    t = _RE_ASSIN.sub(_rep_nome, t)
    return t

def _mascara_ner(vals):
    if not _nlp:
        return list(vals)
    out = []
    for s, doc in zip(vals, _nlp.pipe([v or "" for v in vals], batch_size=64)):
        for a, b in sorted(((e.start_char, e.end_char) for e in doc.ents
                            if e.label_ == "PER" and len(e.text.strip()) > 2), reverse=True):
            s = s[:a] + _token("NOME", s[a:b]) + s[b:]
        out.append(s)
    return out

def sanitizar(t):
    """Pipeline completo para uma string (uso avulso / testes)."""
    if not isinstance(t, str) or not t:
        return t
    t = mascara_estruturada(t)
    t = _mascara_ner([t])[0]
    return mascara_nome_heuristica(t)

# --------------------------------------------------- categoria especial - Art.11 / Art.9 (4)
_RE_SENSIVEL = re.compile(r"(?i)\b(?:" + "|".join([
    r"laudo(?:\s+m[eé]dico)?", r"atestado\s+m[eé]dico", r"CID[\-\s]?[A-Z]?\d", r"diagn[oó]stic\w*",
    r"doen[cç]a", r"transtorno", r"depress\w+", r"ansiedade", r"p[aâ]nico", r"autis\w+", r"TDAH",
    r"defici[eê]nci\w+", r"cadeirante", r"gestante", r"gravidez", r"HIV",
    r"c[aâ]ncer", r"quimioterapia", r"psicol[oó]gic\w+", r"psiqui[aá]tric\w+", r"medicament\w+",
    r"religi[aã]\w+", r"evang[eé]lic\w+", r"cat[oó]lic\w+", r"esp[ií]rita", r"umbanda", r"candombl[eé]",
    r"sindicat\w+", r"filia[cç][aã]o\s+(?:partid|sindic)\w+", r"orienta[cç][aã]o\s+sexual",
    r"ra[cç]a", r"etnia", r"cota\s+racial", r"heteroident\w*", r"biom[eé]tric\w+", r"impress[aã]o\s+digital",
]) + r")\b")

# =========================================================================================
if not SANITIZAR:
    print("SANITIZAR=False -> base gerada com dados pessoais EM CLARO (uso interno controlado).")
else:
    assert MODO_SANITIZACAO in ("rotulo", "pseudonimo"), MODO_SANITIZACAO
    _CONTADOR_PII.clear(); _LOG_SENSIVEL.clear()
    _COLS = [c for c in ("assunto", "pergunta", "resposta", "para", "cc") if c in df.columns]

    # (4) quarentena ANTES de mascarar (senao os termos sensiveis somem)
    if QUARENTENA_SENSIVEL:
        _blob = df[["assunto", "pergunta", "resposta", "historico"]].fillna("").agg("\n".join, axis=1)
        _mask = _blob.str.contains(_RE_SENSIVEL, na=False)
        df_quarentena = df[_mask].copy()
        for idx in df_quarentena.index:
            termos = sorted({mt.group(0).lower() for mt in _RE_SENSIVEL.finditer(_blob.loc[idx])})
            _LOG_SENSIVEL.append({"msg_id": df.loc[idx, "msg_id"],
                                  "assunto": str(df.loc[idx, "assunto"])[:80],
                                  "termos": "; ".join(termos)})
        df = df[~_mask].reset_index(drop=True)
        print(f"quarentena (dado sensivel Art.11/Art.9): {len(df_quarentena)} mensagens fora do RAG")

    # (3) papel no lugar da identidade
    df["remetente_nome"] = [
        _token("ATENDENTE", n) if o == "suporte" else _token("USUARIO")
        for n, o in zip(df["remetente_nome"], df["origem"])]
    df["remetente"] = _token("EMAIL")   # ja usado por origem/eh_interno; nao vai para o RAG

    # (1)+(2) texto livre: regex -> NER -> heuristica
    for c in _COLS:
        vals = [mascara_estruturada(v) for v in df[c].fillna("")]
        vals = _mascara_ner(vals)
        df[c] = [mascara_nome_heuristica(v) for v in vals]

    # metricas de tamanho recalculadas apos o mascaramento
    df["n_chars"] = df["resposta"].str.len()
    df["n_palavras"] = df["resposta"].str.split().str.len().fillna(0).astype(int)

    # (5) checagem anti-vazamento
    def _tem_cpf(t):
        t = t or ""
        return bool(re.search(r"\d{3}\.\d{3}\.\d{3}-\d{2}", t)) or any(_cpf_valido(x) for x in re.findall(r"\d{11}", t))
    _RE_EMAIL_CHK = _SIMPLES[0][1]
    _RE_TEL_CHK = _SIMPLES[7][1]
    fugas = {}
    for c in ("assunto", "pergunta", "resposta"):
        for nome, teste in (("email", lambda s: bool(_RE_EMAIL_CHK.search(s or ""))),
                            ("cpf", _tem_cpf),
                            ("telefone", lambda s: bool(_RE_TEL_CHK.search(s or "")))):
            ids = df.loc[df[c].map(teste), "msg_id"].tolist()
            if ids:
                fugas.setdefault(nome, set()).update(ids)

    print(f"sanitizacao: modo={MODO_SANITIZACAO} | "
          f"salt={'env' if os.environ.get('SAL_SANITIZACAO') else '(sessao) ' + SAL_SANITIZACAO}")
    print("ocorrencias mascaradas:", dict(sorted(_CONTADOR_PII.items())) or "(nenhuma)")
    if fugas:
        det = {k: sorted(v)[:10] for k, v in fugas.items()}
        print("PII RESIDUAL:", det)
        if SANITIZAR_ESTRITO:
            raise SystemExit("Sanitizacao incompleta (SANITIZAR_ESTRITO=True). "
                             "Ajuste os regex da secao 4.5 e reexecute antes de exportar.")
    else:
        print("checagem anti-vazamento: OK (sem email/CPF/telefone residual em assunto/pergunta/resposta)")

    _ex = df.loc[df["resposta"].str.contains(r"\[(?:NOME|EMAIL|CPF|RG|RA|TELEFONE|CEP)[:\]]", na=False), "resposta"]
    if len(_ex):
        print("\n--- exemplo apos mascaramento ---\n", _ex.iloc[0][:400])


## 5. Threads

O agrupamento por assunto normalizado (sem `Re:/Enc:/Fw:`) junta os atendimentos que voltaram
varias vezes. Cada thread guarda a pergunta inicial, todas as respostas dadas e o transcript.


In [ ]:
# 5. Threads
def montar_thread(g: pd.DataFrame) -> dict:
    g = g.sort_values("data", kind="stable")
    com_pergunta = g[g["tem_pergunta"]]
    transcript = "\n\n".join(
        (f"[PERGUNTA] {r['pergunta']}\n" if r["tem_pergunta"] else "") +
        f"[RESPOSTA | {r['remetente_nome']}"
        + (f" | {r['data']:%Y-%m-%d %H:%M}]" if pd.notna(r["data"]) else "]")
        + f"\n{r['resposta']}"
        for _, r in g.iterrows())
    return {
        "thread_id": g["thread_id"].iloc[0],
        "assunto": g["assunto"].iloc[0],
        "assunto_norm": g["assunto_norm"].iloc[0],
        "pasta": g["pasta"].iloc[0],
        "n_mensagens": len(g),
        "data_inicio": g["data"].min(),
        "data_fim": g["data"].max(),
        "atendentes": "; ".join(sorted({x for x in g["remetente_nome"] if x})),
        "destinatarios": "; ".join(sorted({x for x in g["para"] if x})[:5]),
        "pergunta": (com_pergunta["pergunta"].iloc[0] if not com_pergunta.empty else ""),
        "resposta": "\n\n---\n\n".join(g["resposta"].tolist()),
        "anexos": "; ".join(sorted({a for s in g["anexos"] for a in str(s).split("; ") if a})[:10]),
        "transcript": transcript,
    }

df_threads = pd.DataFrame([montar_thread(g) for _, g in df.groupby("thread_id", sort=False)])
df_threads["tem_pergunta"] = df_threads["pergunta"].str.len().ge(20)
df_threads = df_threads.sort_values("data_inicio", kind="stable").reset_index(drop=True)
print(f"threads: {len(df_threads)} | com pergunta identificada: {int(df_threads['tem_pergunta'].sum())}")
display(df_threads[["thread_id", "assunto", "n_mensagens", "tem_pergunta"]].head(8))


## 5.5 Modelos de resposta padrao (respostas repetidas)

Boa parte dos `Itens Enviados` do suporte sao **respostas de catalogo**: o mesmo texto
(instrucoes de matricula, reset de senha, prazo de analise, orientacao de documento) reenviado
dezenas de vezes, mudando so o nome, a data ou um numero — que a **secao 4.5 ja mascarou**.

Aqui essas respostas viram **um unico documento** `tipo = "modelo"`:

1. **Esqueleto** — cada resposta e reduzida ao que nao varia (removidos `[TAG]`, numeros, datas,
   pontuacao). Respostas com o mesmo esqueleto = mesmo modelo.
2. **Merge fuzzy** — esqueletos com similaridade >= `SIMILARIDADE_MODELO`, dentro do mesmo
   assunto, sao unidos (pega variacoes de saudacao/ordem de frases).
3. **Canonico + slots** — o texto mais completo do cluster vira `texto_modelo`, com os campos
   variaveis marcados como `{{nome}}`, `{{data_nasc}}`, `{{protocolo}}`… (a partir das tags da 4.5).
4. Clusters com `>= MIN_OCORRENCIAS_MODELO` respostas entram como modelo; as mensagens individuais
   desse cluster **sem pergunta** (comunicados em massa) saem do indice — o modelo as cobre.
   As que **tem pergunta** continuam como `qa`, so que marcadas com `modelo_id` para o agente
   deduplicar/priorizar a versao canonica.

Resultado para o agente: em vez de recuperar 30 variantes ruidosas do mesmo e-mail, ele recupera
**o modelo oficial** + os pares pergunta->resposta reais, com a contagem de uso como sinal de
confianca.


In [ ]:
# 5.5 Modelos de resposta padrao
from difflib import SequenceMatcher
from collections import defaultdict

_RE_TAG    = re.compile(r"\[[A-Z_]+(?::[0-9a-f]+)?\]")      # [NOME], [CPF:ab12cd] (saida da 4.5)
_RE_DATA   = re.compile(r"\b\d{1,2}[/.\-]\d{1,2}(?:[/.\-]\d{2,4})?\b")
_RE_NUM    = re.compile(r"\d+")
_RE_NAOALF = re.compile(r"[^a-z0-9áàâãéêíóôõúç\s]")
_RE_ESP    = re.compile(r"\s+")

def _esqueleto(t: str) -> str:
    """Reduz a resposta ao que NAO varia entre um envio e outro."""
    t = (t or "").lower()
    t = _RE_TAG.sub(" ", t)
    t = _RE_DATA.sub(" ", t)
    t = _RE_NUM.sub(" ", t)
    t = _RE_NAOALF.sub(" ", t)
    return _RE_ESP.sub(" ", t).strip()

def _placeholders(t: str) -> str:
    """Transforma o que varia entre envios em campos: [NOME]->{{nome}}, datas->{{data}},
    numeros isolados->{{valor}}. Deixa o modelo pronto p/ o agente preencher."""
    t = _RE_TAG.sub(lambda m: "{{" + m.group(0).strip("[]").split(":")[0].lower() + "}}", t)
    t = _RE_DATA.sub("{{data}}", t)
    t = re.sub(r"(?<![\w{])\d{2,}(?![\w}])", "{{valor}}", t)
    return t

df["modelo_id"] = ""
df_modelos = pd.DataFrame(columns=[
    "modelo_id", "assunto", "n_ocorrencias", "n_threads", "primeira_data", "ultima_data",
    "slots", "assuntos_variantes", "thread_ids", "texto_modelo"])

if len(df):
    _b = df[df["n_chars"] >= 60].copy()
    _b["esq"] = _b["resposta"].map(_esqueleto)
    _b = _b[_b["esq"].str.len() >= 40]

    if len(_b):
        # (1) cluster exato por esqueleto
        _b["cluster"] = _b.groupby("esq", sort=False).ngroup()

        # (2) merge fuzzy: 1 representante por cluster, une os parecidos do mesmo assunto
        reps = (_b.sort_values("n_chars", ascending=False)
                  .drop_duplicates("cluster")[["cluster", "assunto_norm", "esq"]]
                  .to_dict("records"))
        pai = {r["cluster"]: r["cluster"] for r in reps}
        def _find(c):
            while pai[c] != c:
                pai[c] = pai[pai[c]]
                c = pai[c]
            return c
        por_assunto = defaultdict(list)
        for r in reps:
            por_assunto[r["assunto_norm"]].append(r)
        for grupo in por_assunto.values():
            for i in range(len(grupo)):
                for j in range(i + 1, len(grupo)):
                    a, b = grupo[i], grupo[j]
                    if _find(a["cluster"]) == _find(b["cluster"]):
                        continue
                    if SequenceMatcher(None, a["esq"][:800], b["esq"][:800]).ratio() >= SIMILARIDADE_MODELO:
                        pai[_find(b["cluster"])] = _find(a["cluster"])
        _b["cluster"] = _b["cluster"].map(_find)

        # (3) monta os modelos (clusters grandes o suficiente)
        linhas_mod = []
        for cid, g in _b.groupby("cluster", sort=False):
            if len(g) < MIN_OCORRENCIAS_MODELO:
                continue
            canon = _placeholders(g.loc[g["n_chars"].idxmax(), "resposta"]).strip()
            slots = "; ".join(sorted(set(re.findall(r"\{\{(\w+)\}\}", canon))))
            _amode = g["assunto"].mode()
            linhas_mod.append({
                "_cluster": cid,
                "assunto": _amode.iloc[0] if len(_amode) else g["assunto"].iloc[0],
                "n_ocorrencias": len(g),
                "n_threads": g["thread_id"].nunique(),
                "primeira_data": g["data"].min(),
                "ultima_data": g["data"].max(),
                "slots": slots,
                "assuntos_variantes": "; ".join(sorted({a for a in g["assunto"] if a})[:8]),
                "thread_ids": "; ".join(sorted(set(g["thread_id"]))[:15]),
                "texto_modelo": canon,
            })

        if linhas_mod:
            df_modelos = (pd.DataFrame(linhas_mod)
                          .sort_values("n_ocorrencias", ascending=False).reset_index(drop=True))
            df_modelos.insert(0, "modelo_id",
                              ["MOD{:04d}".format(i + 1) for i in range(len(df_modelos))])
            _cl2mod = dict(zip(df_modelos["_cluster"], df_modelos["modelo_id"]))
            _msg2mod = {mid: _cl2mod[cl]
                        for mid, cl in zip(_b["msg_id"], _b["cluster"]) if cl in _cl2mod}
            df["modelo_id"] = df["msg_id"].map(_msg2mod).fillna("")
            df_modelos = df_modelos.drop(columns=["_cluster"])

# --- redirecionamento: modelos de contexto "cronograma" -> Secretaria Academica ---
REDIR_MSG = ("Para informacoes sobre cronograma / calendario academico (datas, prazos e "
             f"etapas), entre em contato com a Secretaria Academica pelo e-mail "
             f"{EMAIL_SECRETARIA}. As datas mudam a cada periodo letivo e sao confirmadas "
             "somente por esse canal.")
_RE_REDIR = re.compile("(?i)(?:" + "|".join(CONTEXTOS_REDIRECIONAR) + ")")
if "redirecionado" not in df_modelos.columns:
    df_modelos["redirecionado"] = False
_MODELOS_REDIR = set()
if len(df_modelos):
    _hit = (df_modelos[["assunto", "assuntos_variantes", "texto_modelo"]]
            .fillna("").agg(" ".join, axis=1).str.contains(_RE_REDIR, na=False))
    df_modelos["redirecionado"] = _hit
    df_modelos.loc[_hit, "texto_modelo"] = REDIR_MSG
    df_modelos.loc[_hit, "slots"] = ""
    _MODELOS_REDIR = set(df_modelos.loc[_hit, "modelo_id"])
    if _hit.any():
        print(f"redirecionados p/ Secretaria ({EMAIL_SECRETARIA}): {int(_hit.sum())} modelos "
              f"-> {sorted(_MODELOS_REDIR)}")

_cob = int((df["modelo_id"] != "").sum())
print(f"modelos de resposta: {len(df_modelos)} | cobrem {_cob} mensagens "
      f"({_cob / max(len(df), 1):.0%})")
if len(df_modelos):
    display(df_modelos[["modelo_id", "assunto", "n_ocorrencias", "n_threads", "slots"]].head(10))
    print("\n--- exemplo de modelo ---\n", df_modelos["texto_modelo"].iloc[0][:600])


## 6. Chunking

O documento base e **uma mensagem**: pergunta do usuario + resposta do suporte, que e a unidade
que responde a uma consulta do agente. Mensagens sem pergunta identificada viram documentos de
tipo `resposta` (comunicados, instrucoes enviadas em massa) — ainda uteis como referencia.
As respostas de catalogo detectadas na secao 5.5 entram uma unica vez como `tipo = "modelo"`.


In [ ]:
# 6. Chunking
def dividir(texto, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    texto = (texto or "").strip()
    if len(texto) <= size:
        return [texto] if texto else []
    chunks, ini = [], 0
    while ini < len(texto):
        fim = min(ini + size, len(texto))
        if fim < len(texto):
            meio = ini + size // 2
            corte = max(texto.rfind("\n\n", meio, fim),
                        texto.rfind(". ", meio, fim),
                        texto.rfind("\n", meio, fim))
            if corte > ini:
                fim = corte + 1
        chunks.append(texto[ini:fim].strip())
        if fim >= len(texto):
            break
        ini = max(fim - overlap, ini + 1)
    return [c for c in chunks if len(c) >= MIN_CHARS_CHUNK]

_COLS_CHUNK = ["chunk_id", "msg_id", "thread_id", "tipo", "parte", "assunto", "pasta", "data",
               "ano", "atendente", "destinatario", "n_anexos", "n_chars", "tokens_aprox",
               "modelo_id", "n_ocorrencias", "redirecionado", "texto"]

def _linha_chunk(**kw):
    kw.setdefault("modelo_id", ""); kw.setdefault("n_ocorrencias", 0)
    kw.setdefault("redirecionado", False)
    return {c: kw.get(c) for c in _COLS_CHUNK}

linhas = []

# 6a. Modelos de resposta padrao -> 1 documento canonico cada (tipo="modelo")
for _, md in df_modelos.iterrows():
    _nt = md["n_threads"]
    cabec = (f"Modelo de resposta padrao do suporte "
             f"(usado {md['n_ocorrencias']}x em {_nt} thread{'s' if _nt != 1 else ''}).")
    corpo = (f"Assunto tipico: {md['assunto']}\n"
             + (f"Campos a preencher: {md['slots']}\n" if md["slots"] else "")
             + f"\n{md['texto_modelo']}")
    ano = md["ultima_data"].year if pd.notna(md["ultima_data"]) else None
    for i, ch in enumerate(dividir(corpo), start=1):
        texto = f"{cabec}\n\n{ch}"
        linhas.append(_linha_chunk(
            chunk_id=f"{md['modelo_id']}-{i:03d}", msg_id="", thread_id="", tipo="modelo",
            parte=i, assunto=md["assunto"], pasta="", data=md["ultima_data"], ano=ano,
            atendente="[ATENDENTE]", destinatario="", n_anexos=0,
            modelo_id=md["modelo_id"], n_ocorrencias=int(md["n_ocorrencias"]),
            redirecionado=bool(md.get("redirecionado", False)),
            n_chars=len(texto), tokens_aprox=round(len(texto) / 4), texto=texto))

# 6b. Mensagens individuais
_esq_vistos = set()
for _, m in df.iterrows():
    mod = m.get("modelo_id", "") or ""
    cabec = f"Assunto: {m['assunto']}" if m["assunto"].strip() else "Assunto: (sem assunto no PST)"
    if m["tem_pergunta"]:
        tipo = "qa"
        _resp = REDIR_MSG if mod in _MODELOS_REDIR else m["resposta"]
        corpo = f"Pergunta do usuario:\n{m['pergunta']}\n\nResposta do suporte:\n{_resp}"
    else:
        if mod:                                  # comunicado em massa: o modelo ja cobre
            continue
        _e = _esqueleto(m["resposta"])[:400]      # colapsa respostas identicas abaixo do limiar
        if _e in _esq_vistos:
            continue
        _esq_vistos.add(_e)
        tipo = "resposta"
        corpo = m["resposta"]
    ano = m["data"].year if pd.notna(m["data"]) else None
    for i, ch in enumerate(dividir(corpo), start=1):
        texto = f"{cabec}\n\n{ch}"
        linhas.append(_linha_chunk(
            chunk_id=f"{m['msg_id']}-{i:03d}", msg_id=m["msg_id"], thread_id=m["thread_id"],
            tipo=tipo, parte=i, assunto=m["assunto"], pasta=m["pasta"], data=m["data"], ano=ano,
            atendente=m["remetente_nome"], destinatario=m["para"], n_anexos=m["n_anexos"],
            modelo_id=mod, n_ocorrencias=0, redirecionado=(mod in _MODELOS_REDIR),
            n_chars=len(texto), tokens_aprox=round(len(texto) / 4), texto=texto))

df_chunks = pd.DataFrame(linhas, columns=_COLS_CHUNK)
total_tokens = int(df_chunks["tokens_aprox"].sum()) if len(df_chunks) else 0
print(f"chunks: {len(df_chunks)} | tokens aprox.: {total_tokens}")
if len(df_chunks):
    display(df_chunks["tipo"].value_counts().to_frame("chunks"))
    _amostra = df_chunks[df_chunks.tipo == "modelo"]["texto"]
    if not len(_amostra):
        _amostra = df_chunks[df_chunks.tipo == "qa"]["texto"]
    print("\n--- amostra ---\n", _amostra.iloc[0][:700])


In [ ]:
# 7. Export para Excel
LIMITE_CELULA = 32000  # limite do Excel: 32767 caracteres por celula

def _celula_excel(v):
    """Prepara um valor para o openpyxl: tira timezone, trunca texto e remove chars de controle."""
    if getattr(v, "tzinfo", None) is not None:          # Timestamp/datetime tz-aware -> naive
        return v.replace(tzinfo=None)
    if isinstance(v, str):
        if len(v) > LIMITE_CELULA:
            v = v[:LIMITE_CELULA] + " [...truncado]"
        return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", v)
    return v

def preparar(d):
    d = d.copy()
    for c in d.columns:
        if pd.api.types.is_datetime64_any_dtype(d[c]):
            if getattr(d[c].dt, "tz", None) is not None:
                d[c] = d[c].dt.tz_localize(None)
        elif d[c].dtype == object:                      # colunas mistas (datas tz-aware + NaT, texto)
            d[c] = d[c].map(_celula_excel)
    return d

cols_msg = ["msg_id", "thread_id", "pasta", "assunto", "origem", "remetente_nome", "remetente",
            "para", "cc", "data", "n_palavras", "n_chars", "n_anexos", "anexos",
            "tem_pergunta", "modelo_id", "pergunta", "resposta"]
df_msg_out = df[[c for c in cols_msg if c in df.columns]]

# --- auditoria da sanitizacao (sem conteudo sensivel) ---
df_pii_resumo = (pd.DataFrame(sorted(_CONTADOR_PII.items()), columns=["tipo", "ocorrencias"])
                 if SANITIZAR else pd.DataFrame(columns=["tipo", "ocorrencias"]))
df_quar_log = (pd.DataFrame(_LOG_SENSIVEL) if _LOG_SENSIVEL
               else pd.DataFrame(columns=["msg_id", "assunto", "termos"]))

# mensagens em quarentena vao para arquivo SEPARADO (contem PII em claro; NAO e o RAG)
if SANITIZAR and len(df_quarentena):
    _quar_cols = [c for c in ["msg_id", "pasta", "data", "assunto", "remetente_nome", "remetente",
                              "para", "pergunta", "resposta", "historico"] if c in df_quarentena.columns]
    _quar_path = OUT_XLSX.with_name("quarentena_revisao_manual.csv")
    df_quarentena[_quar_cols].to_csv(_quar_path, index=False, encoding="utf-8-sig")
    print(f"ATENCAO: {len(df_quarentena)} mensagens com possivel dado sensivel -> "
          f"{_quar_path.name} (revisar com o/a DPO; NAO indexar sem tratamento)")

df_estat = pd.DataFrame({
    "metrica": ["arquivo_pst", "motor_extracao", "gerado_em", "mensagens_brutas", "mensagens_uteis",
                "threads", "mensagens_com_pergunta", "modelos_resposta", "chunks_modelo", "chunks_qa",
                "chunks", "tokens_aprox", "chunk_size", "chunk_overlap", "sanitizado",
                "modo_sanitizacao", "mensagens_quarentena", "pii_mascarada",
                "periodo_inicio", "periodo_fim"],
    "valor": [PST_PATH.name, motor, datetime.now().strftime("%Y-%m-%d %H:%M:%S"), len(df_raw), len(df),
              len(df_threads), int(df["tem_pergunta"].sum()), len(df_modelos),
              int((df_chunks["tipo"] == "modelo").sum()) if len(df_chunks) else 0,
              int((df_chunks["tipo"] == "qa").sum()) if len(df_chunks) else 0,
              len(df_chunks), total_tokens, CHUNK_SIZE, CHUNK_OVERLAP,
              SANITIZAR, (MODO_SANITIZACAO if SANITIZAR else "-"), len(df_quarentena),
              (str(dict(sorted(_CONTADOR_PII.items()))) if SANITIZAR else "-"),
              str(df["data"].min()), str(df["data"].max())],
})

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    preparar(df_chunks).to_excel(xw, sheet_name="chunks_rag", index=False)
    preparar(df_modelos).to_excel(xw, sheet_name="modelos_resposta", index=False)
    preparar(df_threads).to_excel(xw, sheet_name="threads", index=False)
    preparar(df_msg_out).to_excel(xw, sheet_name="mensagens", index=False)
    df_estat.to_excel(xw, sheet_name="estatisticas", index=False)
    df_pii_resumo.to_excel(xw, sheet_name="sanitizacao_pii", index=False)
    preparar(df_quar_log).to_excel(xw, sheet_name="quarentena_log", index=False)

    larguras = {
        "chunks_rag":       {"A": 16, "F": 45, "R": 100},
        "modelos_resposta": {"A": 10, "B": 45, "G": 40, "J": 100},
        "threads":          {"A": 16, "B": 45, "K": 60, "L": 60, "N": 100},
        "mensagens":        {"A": 10, "D": 45, "Q": 60, "R": 60},
        "estatisticas":     {"A": 22, "B": 40},
        "sanitizacao_pii":  {"A": 18, "B": 14},
        "quarentena_log":   {"A": 10, "B": 50, "C": 60},
    }
    for nome, cols in larguras.items():
        ws = xw.sheets[nome]
        for col, w in cols.items():
            ws.column_dimensions[col].width = w
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print("Excel gerado:", OUT_XLSX, f"({OUT_XLSX.stat().st_size/1024:.1f} KB)")
if SANITIZAR:
    print("PII mascarada:", dict(sorted(_CONTADOR_PII.items())) or "(nenhuma)")
display(df_estat)


In [ ]:
# 7.1 Tabela SEPARADA so com as tabulacoes (modelos de resposta) em chunks -> para o agente
OUT_MODELOS_XLSX = (BASE_DIR / "modelos_resposta_chunks.xlsx").resolve()

_meta_mod = df_modelos.set_index("modelo_id")[
    ["slots", "assuntos_variantes", "n_threads", "primeira_data", "ultima_data"]] \
    if len(df_modelos) else pd.DataFrame(
        columns=["slots", "assuntos_variantes", "n_threads", "primeira_data", "ultima_data"])

df_chunks_modelos = (
    df_chunks[df_chunks["tipo"] == "modelo"]
    .drop(columns=["msg_id", "thread_id", "pasta", "atendente", "destinatario", "n_anexos", "data"])
    .merge(_meta_mod, left_on="modelo_id", right_index=True, how="left")
    .loc[:, ["chunk_id", "modelo_id", "tipo", "parte", "assunto", "assuntos_variantes",
             "slots", "redirecionado", "n_ocorrencias", "n_threads", "primeira_data",
             "ultima_data", "ano", "n_chars", "tokens_aprox", "texto"]]
    .reset_index(drop=True)
)

with pd.ExcelWriter(OUT_MODELOS_XLSX, engine="openpyxl") as xw:
    preparar(df_chunks_modelos).to_excel(xw, sheet_name="chunks", index=False)   # 1 linha = 1 chunk
    preparar(df_modelos).to_excel(xw, sheet_name="modelos", index=False)         # template inteiro
    for nome, larg in {"chunks": {"A": 16, "E": 40, "G": 35, "P": 100},
                       "modelos": {"A": 10, "B": 45, "G": 35, "J": 100}}.items():
        ws = xw.sheets[nome]
        for col, w in larg.items():
            ws.column_dimensions[col].width = w
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print(f"tabela de modelos: {OUT_MODELOS_XLSX}  "
      f"({len(df_chunks_modelos)} chunks / {len(df_modelos)} modelos)")
if len(df_chunks_modelos):
    display(df_chunks_modelos[["chunk_id", "assunto", "slots", "n_ocorrencias"]].head(10))

# consumo no agente:
#   m = pd.read_excel("modelos_resposta_chunks.xlsx", sheet_name="chunks")
#   collection.add(ids=m["chunk_id"].tolist(),
#                  documents=m["texto"].tolist(),
#                  metadatas=m.drop(columns=["texto"]).to_dict("records"))


In [ ]:
# 7.2 Adicionar modelo(s) manualmente aos Excel ja gerados
# -------------------------------------------------------------------------------------------
# Edite a lista MODELOS_MANUAIS abaixo e execute esta celula (precisa das celulas 5.5, 6 e 7
# ja executadas). E re-executavel: substitui os modelos manuais (prefixo "MODM") da rodada
# anterior, nao acumula. Reescreve base_conhecimento_suporte.xlsx e modelos_resposta_chunks.xlsx.
# -------------------------------------------------------------------------------------------

MODELOS_MANUAIS = [
    {
        "assunto": "Exemplo - troque por um modelo real",
        "texto_modelo": (
            "Prezado(a) {{nome}},\n\n"
            "Descreva aqui o procedimento padrao. Use {{campo}} para partes variaveis "
            "(ex.: {{protocolo}}, {{data}}).\n\n"
            "Atenciosamente,\nSecretaria Academica"
        ),
        # opcionais:
        "slots": "",                 # vazio -> derivado dos {{...}} do texto
        "assuntos_variantes": "",    # sinonimos de assunto separados por ';'
        "n_ocorrencias": 0,          # 0 = modelo curado manualmente
        "redirecionado": False,      # True = tratar como encaminhamento
    },
]

_PREF = "MODM"

def _modelo_manual(d, i):
    txt = str(d["texto_modelo"]).strip()
    slots = (d.get("slots") or "").strip() or "; ".join(sorted(set(re.findall(r"\{\{(\w+)\}\}", txt))))
    return {
        "modelo_id": f"{_PREF}{i:04d}",
        "assunto": str(d.get("assunto", "")).strip(),
        "n_ocorrencias": int(d.get("n_ocorrencias", 0)),
        "n_threads": int(d.get("n_threads", 0)),
        "primeira_data": d.get("primeira_data", pd.NaT),
        "ultima_data": d.get("ultima_data", pd.NaT),
        "slots": slots,
        "assuntos_variantes": str(d.get("assuntos_variantes", "")),
        "thread_ids": str(d.get("thread_ids", "")),
        "texto_modelo": txt,
        "redirecionado": bool(d.get("redirecionado", False)),
    }

assert all(str(d.get("texto_modelo", "")).strip() for d in MODELOS_MANUAIS), \
    "todo modelo precisa de 'texto_modelo'"

# 1) tira os manuais anteriores e reanexa
if "redirecionado" not in df_modelos.columns:
    df_modelos["redirecionado"] = False
df_modelos = df_modelos[~df_modelos["modelo_id"].astype(str).str.startswith(_PREF)].reset_index(drop=True)
df_chunks = df_chunks[~df_chunks["modelo_id"].astype(str).str.startswith(_PREF)].reset_index(drop=True)

_novos = pd.DataFrame([_modelo_manual(d, i) for i, d in enumerate(MODELOS_MANUAIS, 1)])
_novos = _novos.reindex(columns=df_modelos.columns)
df_modelos = pd.concat([df_modelos, _novos], ignore_index=True)
_MODELOS_REDIR = _MODELOS_REDIR | set(_novos.loc[_novos["redirecionado"], "modelo_id"])

# 2) chunks tipo="modelo" dos novos
_linhas_m = []
for _, md in _novos.iterrows():
    cab = ("Modelo de resposta padrao (curado manualmente)."
           if int(md["n_ocorrencias"]) == 0
           else f"Modelo de resposta padrao do suporte (usado {int(md['n_ocorrencias'])}x).")
    corpo = (f"Assunto tipico: {md['assunto']}\n"
             + (f"Campos a preencher: {md['slots']}\n" if md["slots"] else "")
             + f"\n{md['texto_modelo']}")
    for i, ch in enumerate(dividir(corpo), start=1):
        texto = f"{cab}\n\n{ch}"
        _linhas_m.append(_linha_chunk(
            chunk_id=f"{md['modelo_id']}-{i:03d}", msg_id="", thread_id="", tipo="modelo",
            parte=i, assunto=md["assunto"], pasta="", data=pd.NaT, ano=None,
            atendente="[ATENDENTE]", destinatario="", n_anexos=0,
            modelo_id=md["modelo_id"], n_ocorrencias=int(md["n_ocorrencias"]),
            redirecionado=bool(md["redirecionado"]),
            n_chars=len(texto), tokens_aprox=round(len(texto) / 4), texto=texto))
df_chunks = pd.concat([df_chunks, pd.DataFrame(_linhas_m, columns=_COLS_CHUNK)], ignore_index=True)
total_tokens = int(df_chunks["tokens_aprox"].sum())

# 3) reexporta os dois arquivos (mesmas abas das celulas 7 e 7.1)
df_msg_out = df[[c for c in cols_msg if c in df.columns]]
df_estat.loc[df_estat["metrica"] == "modelos_resposta", "valor"] = len(df_modelos)
df_estat.loc[df_estat["metrica"] == "chunks_modelo", "valor"] = int((df_chunks["tipo"] == "modelo").sum())
df_estat.loc[df_estat["metrica"] == "chunks_qa", "valor"] = int((df_chunks["tipo"] == "qa").sum())
df_estat.loc[df_estat["metrica"] == "chunks", "valor"] = len(df_chunks)
df_estat.loc[df_estat["metrica"] == "tokens_aprox", "valor"] = total_tokens

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    preparar(df_chunks).to_excel(xw, sheet_name="chunks_rag", index=False)
    preparar(df_modelos).to_excel(xw, sheet_name="modelos_resposta", index=False)
    preparar(df_threads).to_excel(xw, sheet_name="threads", index=False)
    preparar(df_msg_out).to_excel(xw, sheet_name="mensagens", index=False)
    df_estat.to_excel(xw, sheet_name="estatisticas", index=False)
    df_pii_resumo.to_excel(xw, sheet_name="sanitizacao_pii", index=False)
    preparar(df_quar_log).to_excel(xw, sheet_name="quarentena_log", index=False)
    for nome, cols in {"chunks_rag": {"A": 16, "F": 45, "R": 100},
                       "modelos_resposta": {"A": 10, "B": 45, "G": 40, "J": 100},
                       "threads": {"A": 16, "B": 45, "K": 60, "L": 60, "N": 100},
                       "mensagens": {"A": 10, "D": 45, "Q": 60, "R": 60}}.items():
        ws = xw.sheets[nome]
        for col, w in cols.items():
            ws.column_dimensions[col].width = w
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

_meta_mod = df_modelos.set_index("modelo_id")[
    ["slots", "assuntos_variantes", "n_threads", "primeira_data", "ultima_data"]]
df_chunks_modelos = (
    df_chunks[df_chunks["tipo"] == "modelo"]
    .drop(columns=["msg_id", "thread_id", "pasta", "atendente", "destinatario", "n_anexos", "data"])
    .merge(_meta_mod, left_on="modelo_id", right_index=True, how="left")
    .loc[:, ["chunk_id", "modelo_id", "tipo", "parte", "assunto", "assuntos_variantes",
             "slots", "redirecionado", "n_ocorrencias", "n_threads", "primeira_data",
             "ultima_data", "ano", "n_chars", "tokens_aprox", "texto"]]
    .reset_index(drop=True))
with pd.ExcelWriter(OUT_MODELOS_XLSX, engine="openpyxl") as xw:
    preparar(df_chunks_modelos).to_excel(xw, sheet_name="chunks", index=False)
    preparar(df_modelos).to_excel(xw, sheet_name="modelos", index=False)
    for nome, larg in {"chunks": {"A": 16, "E": 40, "G": 35, "P": 100},
                       "modelos": {"A": 10, "B": 45, "G": 35, "J": 100}}.items():
        ws = xw.sheets[nome]
        for col, w in larg.items():
            ws.column_dimensions[col].width = w
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

print(f"{len(_novos)} modelo(s) manual(is) adicionado(s): {list(_novos['modelo_id'])}")
print(f"base: {OUT_XLSX}  |  modelos: {OUT_MODELOS_XLSX}")
display(df_modelos[df_modelos["modelo_id"].str.startswith(_PREF)]
        [["modelo_id", "assunto", "slots", "redirecionado", "texto_modelo"]])


## 8. Como o RAG consome este arquivo

```python
import pandas as pd
df = pd.read_excel("base_conhecimento_suporte.xlsx", sheet_name="chunks_rag")

docs  = df["texto"].tolist()
ids   = df["chunk_id"].tolist()
metas = df.drop(columns=["texto"]).to_dict("records")   # filtros: tipo, ano, modelo_id, thread_id
# collection.add(ids=ids, documents=docs, metadatas=metas)  # Chroma / pgvector / Qdrant
```

Tipos de chunk (coluna `tipo`):

| `tipo` | O que e | Uso pelo agente |
|---|---|---|
| **`modelo`** | resposta padrao do suporte, deduplicada, com `{{campos}}` e `n_ocorrencias` | fonte primaria para "como respondo X"; confianca ~ `n_ocorrencias` |
| **`qa`** | par real pergunta do usuario -> resposta do suporte | formulacoes reais; se vier com `modelo_id`, o canonico esta no `modelo` |
| **`resposta`** | comunicado/instrucao sem pergunta e sem virar modelo | referencia complementar |

### Arquivos gerados

| Arquivo | Conteudo |
|---|---|
| `base_conhecimento_suporte.xlsx` | base completa: `chunks_rag`, `modelos_resposta`, `threads`, `mensagens`, auditoria |
| **`modelos_resposta_chunks.xlsx`** | so as tabulacoes em chunks — aba `chunks` (1 linha = 1 chunk) + aba `modelos` (template inteiro) |
| `quarentena_revisao_manual.csv` | mensagens com dado sensivel retiradas do RAG (revisao do/a DPO) |

Recomendacoes:

- Para um agente so de respostas padrao, indexe direto `modelos_resposta_chunks.xlsx` (aba `chunks`).
- Na base completa, deduplique a recuperacao por `modelo_id` e priorize `tipo="modelo"` por `n_ocorrencias`.
- Modelos de **cronograma** tem o texto substituido pela orientacao de contatar a Secretaria
  Academica (`puc.digital@puc-campinas.edu.br`); a coluna `redirecionado` marca esses casos.
- Use `modelos_resposta` / aba `modelos` para o agente preencher `{{nome}}`, `{{protocolo}}`… na resposta final.
- Sanitizacao (secao 4.5): confira `estatisticas` (`sanitizado`, `pii_mascarada`, `mensagens_quarentena`)
  antes de subir a base. `MODO_SANITIZACAO="rotulo"` = anonimizacao.
- O prompt do agente deve nunca solicitar nem repetir CPF/RG/RA — responder com o procedimento e
  encaminhar ao canal oficial quando o dado for necessario.
